In [4]:
import deeplabcut
import pandas as pd
from pathlib import Path
from deeplabcut.utils import auxiliaryfunctions

PROJECT = Path(r"dlc\pupil-dani-2026-11-13")
config_path = str(PROJECT / "config.yaml")

cfg = auxiliaryfunctions.read_config(config_path)
merged = pd.read_hdf(next((PROJECT / auxiliaryfunctions.get_training_set_folder(cfg)).glob("CollectedData_*.h5")))
test_inds = [i for i, idx in enumerate(merged.index) if idx[1] == "110535"]
train_inds = [i for i, idx in enumerate(merged.index) if idx[1] != "110535"]
print(len(train_inds), len(test_inds))   # must print 354 39

deeplabcut.create_training_dataset(
    config_path, Shuffles=[2],
    trainIndices=[train_inds], testIndices=[test_inds],
    net_type="resnet_50",
)

354 39


ValueError: Cannot create shuffle 2 as it already exists - you must either create the dataset with `userfeedback=False` or delete the shuffle with index 2 manually (in `dlc-models`/`dlc-models-pytorch` and in the `training-datasets` folder) if you want to create a new shuffle with that index. You can otherwise create a shuffle with a new index. Existing indices are [1, 2, 99, 100].

In [ ]:
deeplabcut.train_network(config_path, shuffle=2)

Training with configuration:
config_version: 0
model: backbone={'type': 'ResNet', 'model_name': 'resnet50_gn', 'output_stride': 16, 'freeze_bn_stats': False, 'freeze_bn_weights': False} heads={'bodypart': {'type': 'HeatmapHead', 'weight_init': 'normal', 'predictor': {'type': 'HeatmapPredictor', 'apply_sigmoid': False, 'clip_scores': True, 'location_refinement': True, 'locref_std': 7.2801}, 'target_generator': {'type': 'HeatmapGaussianGenerator', 'num_heatmaps': 8, 'pos_dist_thresh': 17, 'heatmap_mode': 'KEYPOINT', 'gradient_masking': False, 'generate_locref': True, 'locref_std': 7.2801}, 'criterion': {'heatmap': {'type': 'WeightedMSECriterion', 'weight': 1.0}, 'locref': {'type': 'WeightedHuberCriterion', 'weight': 0.05}}, 'heatmap_config': {'channels': [2048, 8], 'kernel_size': [3], 'strides': [2]}, 'locref_config': {'channels': [2048, 16], 'kernel_size': [3], 'strides': [2]}}} backbone_output_channels=2048 neck=None pose_model=None
net_type: resnet_50
method: bu
device: auto
metadata:

In [ ]:
deeplabcut.evaluate_network(
    config_path, Shuffles=[2],
    snapshots_to_evaluate=["snapshot-100"],
    per_keypoint_evaluation=True,
)